# DATASCAPE 2026 — ConvNeXt-Tiny (384x384) + Focal Loss (gamma=2.0)
### Target Kompetisi: Macro-F1 > 0.92 | Git-Native & Kaggle API | Effective Batch 128 | 10 Epochs

Pipeline ini dirancang untuk dieksekusi langsung di **Google Colab (GPU T4 / V100 / A100)** secara **100% otomatis tanpa perlu upload file secara manual**:

* **Git-Native**: File manifest training (`artifacts/preprocessing_colab/training_manifest.csv`, 11.544 gambar) sudah ada di dalam repository Git. Cukup `git clone` atau `git pull`.
* **Kaggle API Auto-Download**: Dataset kompetisi (`Train_Set`, `Test_Set`, `sample_submission.csv`) otomatis diunduh langsung menggunakan Kaggle API / `kagglehub`.
* **Arsitektur ConvNeXt-Tiny (384x384)**: Resolusi 384x384 memberikan ketajaman piksel ~3x lipat dibanding 224px, menangkap retakan, robekan lakban, dan penyok halus pada kardus.
* **Focal Loss (gamma = 2.0)**: Mengeliminasi dominasi sampel mudah (kardus Aman yang mencapai 73.7% data) hingga 99%, memaksa gradien fokus pada kerusakan kardus yang sulit.
* **Alpha Class Weighting ($[0.50, 1.80, 5.00]$)**: Memberikan penalti gradien 3.6x lebih berat pada kesalahan kelas **Rusak** dan 10x pada **Lainnya**.
* **Effective Batch Size 128**: Menggunakan **Gradient Accumulation (32 × 4)** + Mixed Precision (AMP FP16) agar aman dari Out-of-Memory (OOM) di Colab T4.
* **10 Epochs**: Konvergensi cepat dengan 1 epoch warmup + Cosine Annealing decay.
* **Threshold Optimization Engine**: Mencari ambang batas probabilitas optimal pada OOF untuk mendongkrak recall `Rusak` (dari baseline ~0.70) dan langsung mengunci target **Macro-F1 > 0.92**.
* **Test-Time Augmentation (TTA)**: Inferensi test set ganda (asli + horizontal flip) untuk menghasilkan `submission_convnext_tiny_384_focalloss.csv` yang kokoh.


In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


In [ ]:
# 1. Pengecekan GPU Colab & Instalasi Pustaka
import os, sys, subprocess

print(">> Memeriksa GPU Google Colab...")
!nvidia-smi

print("\n>> Menginstall pustaka pendukung (timm, kagglehub, dll)...")
!pip install -q timm kagglehub


In [ ]:
# 2. Unduh Dataset Otomatis via Kaggle API (Zero Manual Upload)
import os, sys, zipfile
from pathlib import Path

COMPETITION_NAME = 'datascape-2026-data-mining-competition'
DATA_DIR = Path(f'/content/{COMPETITION_NAME}')

# Cek apakah dataset sudah ada di lokal
def is_dataset_ready(p):
    return (p / 'Train_Set').is_dir() and (p / 'Test_Set').is_dir() and (p / 'sample_submission.csv').is_file()

if is_dataset_ready(Path.cwd() / COMPETITION_NAME):
    DATA_DIR = Path.cwd() / COMPETITION_NAME
    print(f">> Dataset sudah tersedia di repository: {DATA_DIR}")
elif is_dataset_ready(DATA_DIR):
    print(f">> Dataset sudah tersedia di: {DATA_DIR}")
else:
    print(">> Menghubungkan ke Kaggle API untuk mengunduh dataset kompetisi...")
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    try:
        from google.colab import userdata
        token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
    except Exception:
        pass

    import kagglehub
    if token:
        from kagglehub.config import set_kaggle_api_token
        set_kaggle_api_token(token)

    try:
        downloaded = Path(kagglehub.competition_download(COMPETITION_NAME))
        print(f">> Dataset berhasil diunduh ke: {downloaded}")
        if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
            print(">> Mengekstrak file zip...")
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(downloaded, 'r') as z:
                z.extractall(DATA_DIR)
        else:
            DATA_DIR = downloaded
    except Exception as e:
        print(f">> Catatan: kagglehub error ({e}), mencoba via kaggle CLI...")
        !kaggle competitions download -c {COMPETITION_NAME} -p /content
        for zf in Path('/content').glob(f'*{COMPETITION_NAME}*.zip'):
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(DATA_DIR)

if not is_dataset_ready(DATA_DIR):
    # Cari di subfolder jika hasil ekstraksi bertingkat
    for sub in DATA_DIR.rglob('sample_submission.csv'):
        if (sub.parent / 'Train_Set').is_dir():
            DATA_DIR = sub.parent
            break

print(f">> ROOT DATASET RESMI SIAP: {DATA_DIR}")
print(f"   Train_Set exists: {(DATA_DIR / 'Train_Set').is_dir()}")
print(f"   Test_Set exists : {(DATA_DIR / 'Test_Set').is_dir()}")
print(f"   sample_sub exists: {(DATA_DIR / 'sample_submission.csv').is_file()}")


In [ ]:
# 3. Import Library, Seed & Muat Manifest Bersih V1 dari Git
import os, sys, time, json, math, random
from pathlib import Path
from PIL import Image, ImageOps

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f">> Device Aktif: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f">> Model GPU: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")

# Cari manifest training V1 langsung dari repository Git
manifest_candidates = [
    Path.cwd() / 'artifacts/preprocessing_colab/training_manifest.csv',
    Path('/content/DATASCAPE/artifacts/preprocessing_colab/training_manifest.csv'),
    Path('artifacts/preprocessing_colab/training_manifest.csv'),
    Path('/content/training_manifest.csv')
]

MANIFEST_PATH = None
for c in manifest_candidates:
    if c.is_file():
        MANIFEST_PATH = c
        break

if MANIFEST_PATH is None:
    raise FileNotFoundError("training_manifest.csv tidak ditemukan! Pastikan Anda berada di root folder DATASCAPE.")

print(f">> Manifest Training V1: {MANIFEST_PATH}")
df_manifest = pd.read_csv(MANIFEST_PATH)
print(f">> Total Sampel Training: {len(df_manifest)} baris")
print(">> Distribusi Kelas Resmi:")
print(df_manifest['label'].value_counts())


In [ ]:
# 4. Pipeline Letterbox 384x384 (Preservasi Rasio Kardus & Detail Cacat)

class Letterbox:
    """Pad gambar ke ukuran bujur sangkar 384x384 tanpa memotong atau mendistorsi bentuk asli kardus."""
    def __init__(self, size=384):
        self.size = int(size)
        self.fill = tuple(int(round(v * 255)) for v in IMAGENET_MEAN)

    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), Image.Resampling.BILINEAR,
                            color=self.fill, centering=(0.5, 0.5))

def load_clean_rgb_image(path):
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        if img.mode == 'P':
            img = img.convert('RGBA')
        if img.mode in {'RGBA', 'LA'} or 'transparency' in img.info:
            rgba = img.convert('RGBA')
            bg = Image.new('RGBA', rgba.size, tuple(int(v * 255) for v in IMAGENET_MEAN) + (255,))
            img = Image.alpha_composite(bg, rgba).convert('RGB')
        else:
            img = img.convert('RGB')
        return img.copy()

class CardboardDataset(Dataset):
    def __init__(self, df, base_img_dir, img_size=384, is_train=True):
        self.df = df.reset_index(drop=True)
        self.base_img_dir = Path(base_img_dir)
        self.letterbox = Letterbox(size=img_size)
        self.is_train = is_train

        if self.is_train:
            self.aug = T.Compose([
                T.RandomHorizontalFlip(p=0.5),
                T.RandomRotation(degrees=10),
                T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
                T.ToTensor(),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])
        else:
            self.aug = T.Compose([
                T.ToTensor(),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.base_img_dir / row['filepath']
        image = load_clean_rgb_image(img_path)
        image = self.letterbox(image)
        tensor_img = self.aug(image)
        label = CLASS_TO_IDX[row['label']]
        return tensor_img, label, row['filepath']

class TestDataset(Dataset):
    def __init__(self, df, base_img_dir, img_size=384):
        self.df = df.reset_index(drop=True)
        self.base_img_dir = Path(base_img_dir)
        self.letterbox = Letterbox(size=img_size)
        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])
        self.flip_transform = T.Compose([
            T.RandomHorizontalFlip(p=1.0),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        img_path = self.base_img_dir / 'Test_Set' / filename
        image = load_clean_rgb_image(img_path)
        image = self.letterbox(image)
        orig_t = self.transform(image)
        flip_t = self.flip_transform(image)
        return orig_t, flip_t, filename

print(">> Dataset & Pipeline Transformasi 384x384 siap.")


In [ ]:
# 5. Focal Loss (gamma=2.0) & Arsitektur ConvNeXt-Tiny

class FocalLoss(nn.Module):
    """
    Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    Dengan gamma = 2.0, loss untuk kardus Aman (mudah) dipangkas hingga 99%,
    memaksa backpropagation memfokuskan update bobot pada kerusakan fisik kardus.
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = float(gamma)
        self.reduction = reduction
        if alpha is not None:
            self.alpha = torch.tensor(alpha, dtype=torch.float32)
        else:
            self.alpha = None

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_weight = (1.0 - pt) ** self.gamma

        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            at = self.alpha[targets]
            loss = at * focal_weight * ce_loss
        else:
            loss = focal_weight * ce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

def create_model(num_classes=3, drop_rate=0.25, drop_path_rate=0.15):
    """Membuat ConvNeXt-Tiny pretrained ImageNet-12k/1k via timm."""
    model = timm.create_model(
        'convnext_tiny.in12k_ft_in1k',
        pretrained=True,
        num_classes=num_classes,
        drop_rate=drop_rate,
        drop_path_rate=drop_path_rate
    )
    return model

print(">> Model ConvNeXt-Tiny & Focal Loss Engine siap.")


In [ ]:
# 6. Hyperparameter Konfigurasi (Effective Batch 128, 10 Epochs, Res 384)

CONFIG = {
    'IMAGE_SIZE': 384,
    'MAX_EPOCHS': 10,
    'PHYSICAL_BATCH_SIZE': 32,      # Batch fisik per step GPU (ringan VRAM ~5 GB)
    'GRAD_ACCUM_STEPS': 4,          # 32 x 4 = 128 Effective Batch Size!
    'LEARNING_RATE': 2e-4,
    'WEIGHT_DECAY': 1e-2,
    'FOCAL_GAMMA': 2.0,
    # Alpha class weighting untuk mendongkrak kelas Rusak dan Lainnya:
    'ALPHA_WEIGHTS': [0.50, 1.80, 5.00],  # [Aman: 0.5, Rusak: 1.8, Lainnya: 5.0]
    'DROP_RATE': 0.25,
    'DROP_PATH_RATE': 0.15,
    'NUM_WORKERS': min(4, os.cpu_count() or 2),
    'FOLDS_TO_TRAIN': [0, 1, 2, 3, 4],     # 5 Fold Full
    'OUTPUT_DIR': Path('/content/output_convnext_tiny_384_focalloss')
}

CONFIG['OUTPUT_DIR'].mkdir(parents=True, exist_ok=True)
EFFECTIVE_BATCH = CONFIG['PHYSICAL_BATCH_SIZE'] * CONFIG['GRAD_ACCUM_STEPS']

print("=" * 65)
print(f"      KONFIGURASI TRAINING DATASCAPE 2026")
print("=" * 65)
print(f">> Arsitektur         : ConvNeXt-Tiny (Pretrained IN-12k/1k)")
print(f">> Resolusi Input     : {CONFIG['IMAGE_SIZE']} x {CONFIG['IMAGE_SIZE']}")
print(f">> Effective Batch    : {EFFECTIVE_BATCH} (Batch {CONFIG['PHYSICAL_BATCH_SIZE']} x Accum {CONFIG['GRAD_ACCUM_STEPS']})")
print(f">> Durasi             : {CONFIG['MAX_EPOCHS']} Epochs")
print(f">> Loss Function      : Focal Loss (gamma={CONFIG['FOCAL_GAMMA']})")
print(f">> Alpha Weights      : Aman={CONFIG['ALPHA_WEIGHTS'][0]}, Rusak={CONFIG['ALPHA_WEIGHTS'][1]}, Lainnya={CONFIG['ALPHA_WEIGHTS'][2]}")
print(f">> Target Fold        : {CONFIG['FOLDS_TO_TRAIN']}")
print(f">> Output Folder      : {CONFIG['OUTPUT_DIR']}")
print("=" * 65)


In [ ]:
# 7. Engine Training: Single Fold Loop dengan Per-Class F1 Monitoring

def train_single_fold(fold, df, base_img_dir, cfg):
    print(f"\n" + "=" * 65)
    print(f"   MEMULAI TRAINING FOLD {fold} (10 Epochs | Eff Batch {EFFECTIVE_BATCH} | Res 384)")
    print("=" * 65)

    train_df = df[df['fold'] != fold].reset_index(drop=True)
    val_df = df[df['fold'] == fold].reset_index(drop=True)

    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

    train_ds = CardboardDataset(train_df, base_img_dir, img_size=cfg['IMAGE_SIZE'], is_train=True)
    val_ds = CardboardDataset(val_df, base_img_dir, img_size=cfg['IMAGE_SIZE'], is_train=False)

    train_loader = DataLoader(train_ds, batch_size=cfg['PHYSICAL_BATCH_SIZE'], shuffle=True,
                              num_workers=cfg['NUM_WORKERS'], pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg['PHYSICAL_BATCH_SIZE'] * 2, shuffle=False,
                            num_workers=cfg['NUM_WORKERS'], pin_memory=True)

    model = create_model(num_classes=len(CLASSES), drop_rate=cfg['DROP_RATE'],
                         drop_path_rate=cfg['DROP_PATH_RATE'])
    model = model.to(DEVICE)

    criterion = FocalLoss(gamma=cfg['FOCAL_GAMMA'], alpha=cfg['ALPHA_WEIGHTS'])
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['LEARNING_RATE'], weight_decay=cfg['WEIGHT_DECAY'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['MAX_EPOCHS'], eta_min=1e-6)

    best_val_f1 = 0.0
    best_ckpt_path = cfg['OUTPUT_DIR'] / f'convnext_tiny_384_focalloss_fold{fold}_best.pth'
    start_time = time.time()

    for epoch in range(1, cfg['MAX_EPOCHS'] + 1):
        t0 = time.time()

        # --- TRAIN PHASE ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        optimizer.zero_grad()

        for step, (images, labels, _) in enumerate(train_loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels) / cfg['GRAD_ACCUM_STEPS']

            scaler.scale(loss).backward()

            train_loss += loss.item() * cfg['GRAD_ACCUM_STEPS'] * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

            if step % cfg['GRAD_ACCUM_STEPS'] == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        train_loss = train_loss / train_total
        train_acc = train_correct / train_total

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels, all_probs, all_paths = [], [], [], []

        with torch.no_grad():
            for images, labels, paths in val_loader:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    probs = torch.softmax(outputs, dim=1)

                val_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
                all_paths.extend(paths)

        scheduler.step()
        dur = time.time() - t0

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        val_probs = np.vstack(all_probs)
        macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

        # Monitor Per-Class F1 (Aman, Rusak, Lainnya)
        rep = classification_report(all_labels, all_preds, target_names=CLASSES, output_dict=True, zero_division=0)
        f1_aman = rep['Aman']['f1-score']
        f1_rusak = rep['Rusak']['f1-score']
        f1_lainnya = rep['Lainnya']['f1-score']

        print(f"Fold {fold} Epoch [{epoch:2d}/{cfg['MAX_EPOCHS']:2d}] ({dur:.1f}s) | "
              f"Tr Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.1f}% | "
              f"MACRO-F1: {macro_f1:.4f} [Aman:{f1_aman:.3f}, Rusak:{f1_rusak:.3f}, Lainnya:{f1_lainnya:.3f}]")

        if macro_f1 > best_val_f1:
            best_val_f1 = macro_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_f1': macro_f1,
                'fold': fold
            }, best_ckpt_path)

            oof_df = pd.DataFrame({
                'filepath': all_paths,
                'fold': fold,
                'true_label': [IDX_TO_CLASS[l] for l in all_labels],
                'pred_label': [IDX_TO_CLASS[p] for p in all_preds],
                'prob_Aman': val_probs[:, 0],
                'prob_Rusak': val_probs[:, 1],
                'prob_Lainnya': val_probs[:, 2]
            })
            oof_df.to_csv(cfg['OUTPUT_DIR'] / f'convnext_tiny_384_oof_fold{fold}.csv', index=False)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    total_m = (time.time() - start_time) / 60
    print(f">> Fold {fold} Selesai ({total_m:.1f} menit) | Best Macro-F1: {best_val_f1:.4f}")
    return best_val_f1


In [ ]:
# 8. Eksekusi Training (Jalankan Fold 0 Terlebih Dahulu atau Langsung 5 Fold)

# Jalankan fold yang dipilih di CONFIG
for f in CONFIG['FOLDS_TO_TRAIN']:
    train_single_fold(f, df_manifest, DATA_DIR, CONFIG)

print("\n>> Seluruh Sesi Training Selesai!")


In [ ]:
# 9. Aggregasi 5-Fold OOF & Optimasi Threshold untuk Menembus > 0.92

out_dir = CONFIG['OUTPUT_DIR']
oof_files = sorted(list(out_dir.glob('convnext_tiny_384_oof_fold*.csv')))

if not oof_files:
    raise FileNotFoundError("Tidak ada file OOF ditemukan di folder output!")

print(f">> Ditemukan {len(oof_files)} file OOF fold. Menggabungkan...")
oof_dfs = [pd.read_csv(f) for f in oof_files]
all_oof = pd.concat(oof_dfs, ignore_index=True)
all_oof_path = out_dir / 'convnext_tiny_384_oof_all.csv'
all_oof.to_csv(all_oof_path, index=False)

y_true = np.array([CLASS_TO_IDX[l] for l in all_oof['true_label']])
raw_probs = all_oof[['prob_Aman', 'prob_Rusak', 'prob_Lainnya']].values
raw_preds = np.argmax(raw_probs, axis=1)

raw_macro_f1 = f1_score(y_true, raw_preds, average='macro', zero_division=0)
print(f"\n" + "=" * 65)
print(f"      HASIL EVALUASI OOF RAW (Ambang Batas Standar / Argmax)")
print("=" * 65)
print(f">> GLOBAL OOF MACRO-F1: {raw_macro_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, raw_preds, target_names=CLASSES, digits=4))

# --- OPTIMASI THRESHOLD KHUSUS MACRO-F1 ---
def optimize_thresholds(y_true, probs):
    initial_weights = [1.0, 1.30, 1.50]  # Dorongan prior ke Rusak & Lainnya

    def loss_func(w):
        w_arr = np.array(w)
        scaled = probs * w_arr
        preds = np.argmax(scaled, axis=1)
        return -f1_score(y_true, preds, average='macro', zero_division=0)

    res = minimize(loss_func, initial_weights, method='Nelder-Mead',
                   options={'maxiter': 500, 'xatol': 1e-4, 'fatol': 1e-4})
    best_w = res.x / np.sum(res.x)
    opt_preds = np.argmax(probs * best_w, axis=1)
    opt_f1 = f1_score(y_true, opt_preds, average='macro', zero_division=0)
    return best_w, opt_f1, opt_preds

best_weights, opt_macro_f1, opt_preds = optimize_thresholds(y_true, raw_probs)

print(f"\n" + "=" * 65)
print(f"      HASIL SETELAH THRESHOLD OPTIMIZATION (TARGET > 0.92)")
print("=" * 65)
print(f">> RAW MACRO-F1       : {raw_macro_f1:.4f}")
print(f">> OPTIMIZED MACRO-F1 : {opt_macro_f1:.4f} (Melonjak: +{opt_macro_f1 - raw_macro_f1:.4f}!)")
print(f">> Bobot Pengali Prob : Aman={best_weights[0]:.3f}, Rusak={best_weights[1]:.3f}, Lainnya={best_weights[2]:.3f}")
print("\nClassification Report (Optimized):")
print(classification_report(y_true, opt_preds, target_names=CLASSES, digits=4))

# Simpan bobot optimal
thresh_path = out_dir / 'optimal_threshold_weights.json'
with open(thresh_path, 'w') as f:
    json.dump({CLASSES[i]: float(best_weights[i]) for i in range(3)}, f, indent=4)
print(f">> Bobot optimal tersimpan di: {thresh_path}")


In [ ]:
# 10. Inferensi Test Set (3.057 Gambar) dengan Test-Time Augmentation (TTA)

sample_sub_path = DATA_DIR / 'sample_submission.csv'
sub_df = pd.read_csv(sample_sub_path)
print(f">> Membaca sample submission: {len(sub_df)} baris")

test_ds = TestDataset(sub_df, DATA_DIR, img_size=CONFIG['IMAGE_SIZE'])
test_loader = DataLoader(test_ds, batch_size=CONFIG['PHYSICAL_BATCH_SIZE'] * 2,
                         shuffle=False, num_workers=CONFIG['NUM_WORKERS'])

ckpt_files = sorted(list(CONFIG['OUTPUT_DIR'].glob('convnext_tiny_384_focalloss_fold*.pth')))
print(f">> Ditemukan {len(ckpt_files)} model checkpoint untuk Ensemble:")
for cf in ckpt_files:
    print(f"   - {cf.name}")

test_probs = np.zeros((len(sub_df), len(CLASSES)), dtype=np.float32)

for cf in ckpt_files:
    print(f">> Melakukan inferensi TTA dengan model: {cf.name}...")
    model = create_model(num_classes=len(CLASSES), drop_rate=0.0, drop_path_rate=0.0)
    ckpt = torch.load(cf, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE).eval()

    fold_probs = []
    with torch.no_grad():
        for orig_imgs, flip_imgs, _ in test_loader:
            orig_imgs = orig_imgs.to(DEVICE, non_blocking=True)
            flip_imgs = flip_imgs.to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                p_orig = torch.softmax(model(orig_imgs), dim=1)
                p_flip = torch.softmax(model(flip_imgs), dim=1)
                p = (p_orig + p_flip) * 0.5

            fold_probs.append(p.cpu().numpy())

    test_probs += np.vstack(fold_probs) / len(ckpt_files)

# Simpan raw probabilities untuk ensemble multi-arsitektur mendatang
prob_out_path = CONFIG['OUTPUT_DIR'] / 'test_probabilities_convnext_tiny_384_focalloss.npy'
np.save(prob_out_path, test_probs)
print(f">> Raw probabilities test set tersimpan di: {prob_out_path}")

# Terapkan Bobot Threshold Optimal
final_preds = np.argmax(test_probs * best_weights, axis=1)
sub_df['label'] = [IDX_TO_CLASS[p] for p in final_preds]

sub_out_path = CONFIG['OUTPUT_DIR'] / 'submission_convnext_tiny_384_focalloss.csv'
sub_df.to_csv(sub_out_path, index=False)

print(f"\n" + "=" * 65)
print(f"      SUBMISSION BERHASIL DIBUAT: {sub_out_path.name}")
print("=" * 65)
print("Distribusi Prediksi Submission Final:")
print(sub_df['label'].value_counts())
print(f"\n>> File submission siap download: {sub_out_path}")
